In [0]:
# Funções do Spark (trim, lower, when, col...) com o apelido F
from pyspark.sql import functions as F

# Padrão de nomes do grupo
CATALOG = "vcredit_medallion"
BRONZE_SCHEMA = f"{CATALOG}.bronze"
SILVER_SCHEMA = f"{CATALOG}.silver"

# Tabela onde a silver de chamados será gravada
TABELA_DESTINO = f"{SILVER_SCHEMA}.tb_chamados"

# Coluna de controle 
COLUNA_CONTROLE = "ingestion_datetime"

In [0]:
# Lê a tabela de chamados da bronze do grupo
ch = spark.table(f"{BRONZE_SCHEMA}.tb_chamados")

# Lista só as colunas de dados, deixando de fora a coluna de controle da bronze.
# Se ela entrasse na comparação, duas linhas iguais carregadas em momentos
# diferentes não seriam vistas como duplicatas
colunas_dados = [c for c in ch.columns if c != COLUNA_CONTROLE]

# Contagem antes da limpeza (esperado: 134.474)
linhas_bronze = ch.count()

# Remove as linhas repetidas comparando só as colunas de dados.
# A coluna de controle continua na tabela e segue para a silver
ch = ch.dropDuplicates(colunas_dados)

# Contagem depois da limpeza (esperado: 131.838)
linhas_sem_duplicatas = ch.count()

print(f"bronze: {linhas_bronze}  |  sem duplicatas: {linhas_sem_duplicatas}  |  removidas: {linhas_bronze - linhas_sem_duplicatas}")

bronze: 134474  |  sem duplicatas: 131838  |  removidas: 2636


In [0]:
def chave(col):
    """Reduz um texto a uma chave de comparação."""
    # Tira espaços das pontas e deixa tudo minúsculo
    texto = F.lower(F.trim(col))
    # Troca cada letra acentuada pela versão sem acento (posição a posição nas duas listas)
    texto = F.translate(texto, "áàâãäéèêëíìîïóòôõöúùûüç", "aaaaaeeeeiiiiooooouuuuc")
    # Apaga tudo que não for letra ou número (espaço, ponto, sublinhado, parênteses)
    texto = F.regexp_replace(texto, "[^a-z0-9]", "")
    # Iguala a abreviação: "Atend." e "Atendimento" passam a gerar a mesma chave
    texto = F.regexp_replace(texto, "atendimento", "atend")
    return texto

# Silver de motivos (já com categoria preenchida e criticidade padronizada):
# id, nome oficial, categoria, criticidade e chave do nome oficial
motivos = (
    spark.table(f"{SILVER_SCHEMA}.tb_base_motivos")
    .select(
        "id_motivo",
        F.col("nome_motivo").alias("motivo"),
        "categoria",
        "criticidade",
        chave(F.col("nome_motivo")).alias("chave_motivo"),
    )
)

# Silver de canais: id, nome oficial, tipo (Autosserviço ou Humano) e chave
canais = (
    spark.table(f"{SILVER_SCHEMA}.tb_canais")
    .select(
        "id_canal",
        F.col("nome_canal").alias("canal"),
        "tipo_canal",
        chave(F.col("nome_canal")).alias("chave_canal"),
    )
)

ch = (
    ch
    # Calcula a chave do motivo e do canal digitados em cada chamado
    .withColumn("chave_motivo", chave(F.col("motivo")))
    .withColumn("chave_canal", chave(F.col("canal")))
    # Guarda o texto original com outro nome, para rastreabilidade
    # e para não conflitar com as colunas "motivo" e "canal" das tabelas de referência
    .withColumnRenamed("motivo", "motivo_origem")
    .withColumnRenamed("canal", "canal_origem")
    # Liga cada chamado ao motivo e ao canal oficiais pela chave.
    # Left join para nenhum chamado sumir: se a chave não bater, o motivo fica nulo
    # e a validação no fim do notebook acusa
    .join(motivos, "chave_motivo", "left")
    .join(canais, "chave_canal", "left")
)

In [0]:
# Reaproveita a função chave: "Não" vira "nao", "Nao resolvido" vira "naoresolvido"
resolvido_chave = chave(F.col("resolvido"))

ch = ch.withColumn(
    "resolvido",
    # Valores que indicam sim viram True
    F.when(resolvido_chave.isin("sim", "s", "1"), F.lit(True))
    # Valores que indicam não viram False
     .when(resolvido_chave.isin("nao", "n", "0", "naoresolvido"), F.lit(False))
    # Vazios continuam nulos: não dá para saber se o chamado foi resolvido
     .otherwise(F.lit(None).cast("boolean"))
)

In [0]:
# Lista dos ids que existem na base de atendentes.
# O distinct garante um id por linha, senão o join duplicaria chamados
atendentes_validos = (
    spark.table(f"{BRONZE_SCHEMA}.tb_base_atendentes")
    .select(F.trim("id_atendente").cast("int").alias("id_atendente_valido"))
    .distinct()
)

ch = (
    ch
    # Guarda o id como veio, inclusive os inválidos (0, 99, 999)
    .withColumn("id_atendente_origem", F.trim("id_atendente"))
    # Procura o id na lista de válidos. try_cast devolve nulo se o texto não for número
    .join(
        atendentes_validos,
        F.expr("try_cast(id_atendente_origem as int)") == F.col("id_atendente_valido"),
        "left",
    )
    # Se achou, recebe o id. Se não achou, fica nulo
    .withColumn("id_atendente", F.col("id_atendente_valido"))
    # Marca se o atendente foi identificado (False nos 764 casos inválidos)
    .withColumn("atendente_identificado", F.col("id_atendente").isNotNull())
)

In [0]:
# Lista dos ids que existem na base de clientes
clientes = spark.table(f"{BRONZE_SCHEMA}.tb_clientes").select(F.trim("id_cliente").alias("id_cliente_base")).distinct()

ch = (
    ch
    # Converte os tempos para número. O texto "NULL" não é número, então vira nulo.
    # try_cast é usado porque o Databricks roda em modo ANSI e um cast comum daria erro
    .withColumn("tempo_espera_seg_origem", F.expr("try_cast(tempo_espera as double)"))
    .withColumn("tempo_atendimento_seg_origem", F.expr("try_cast(tempo_atendimento as double)"))
    # O id_cliente não é corrigido aqui, só limpo de espaços
    .withColumn("id_cliente", F.trim("id_cliente"))
    # O join serve apenas para saber se o cliente existe na base de clientes
    .join(clientes, F.col("id_cliente") == F.col("id_cliente_base"), "left")
    # False nos 1.330 casos em que o cliente não existe. Na gold, o cliente certo vem de chamados_hora
    .withColumn("cliente_encontrado", F.col("id_cliente_base").isNotNull())
)

In [0]:
# Escolhe e ordena as colunas finais. O que não está na lista fica de fora:
# os campos de horário (sem data), as chaves auxiliares e as colunas temporárias dos joins
silver = ch.select(
    # id convertido para número, o que deixa os joins da gold mais rápidos
    F.expr("try_cast(id_chamado as bigint)").alias("id_chamado"),
    "id_cliente",
    "cliente_encontrado",
    "id_motivo",
    "motivo",
    "categoria",
    "criticidade",
    "motivo_origem",
    "id_canal",
    "canal",
    "tipo_canal",
    "canal_origem",
    "resolvido",
    "id_atendente",
    "atendente_identificado",
    "id_atendente_origem",
    "tempo_espera_seg_origem",
    "tempo_atendimento_seg_origem",
    # Momento da ingestão na bronze, mantido como nas outras silvers do grupo
    F.col(COLUNA_CONTROLE).cast("timestamp").alias(COLUNA_CONTROLE),
)

(
    silver.write
    # overwrite substitui a tabela inteira a cada execução: rodar de novo não duplica nada
    .mode("overwrite")
    # permite mudar a estrutura da tabela se uma coluna for adicionada ou removida depois
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_DESTINO)
)

print(f"{TABELA_DESTINO} gravada com {spark.table(TABELA_DESTINO).count()} linhas")

vcredit_medallion.silver.tb_chamados gravada com 131838 linhas


In [0]:
# Lê a tabela recém-gravada para conferir o resultado
s = spark.table(TABELA_DESTINO)

# Contagens usadas nas regras abaixo
total = s.count()
ids_distintos = s.select("id_chamado").distinct().count()
sem_motivo = s.filter(F.col("id_motivo").isNull()).count()
sem_canal = s.filter(F.col("id_canal").isNull()).count()
id_invalido = s.filter(F.col("id_chamado").isNull()).count()

# Cada assert testa uma regra. Se ela for falsa, o notebook para com a mensagem de erro
# Regra 1: cada id_chamado aparece uma única vez
assert total == ids_distintos, f"Há id_chamado repetido: {total} linhas para {ids_distintos} ids"
# Regra 2: todos os ids viraram número
assert id_invalido == 0, f"{id_invalido} id_chamado não converteram para número"
# Regra 3: todo chamado tem motivo padronizado (a mensagem lista as grafias não reconhecidas)
assert sem_motivo == 0, f"{sem_motivo} chamados sem motivo padronizado. Grafias não reconhecidas: " + \
    str([r[0] for r in s.filter(F.col('id_motivo').isNull()).select('motivo_origem').distinct().collect()])
# Regra 4: todo chamado tem canal padronizado
assert sem_canal == 0, f"{sem_canal} chamados sem canal padronizado. Grafias não reconhecidas: " + \
    str([r[0] for r in s.filter(F.col('id_canal').isNull()).select('canal_origem').distinct().collect()])

print("Todas as validações passaram")

Todas as validações passaram


In [0]:
# Retrato do tratamento numa linha só.
# Truque usado: booleano convertido para int vira 1 (True) ou 0 (False), então somar conta os True.
# O ~ significa "não"
resumo = s.agg(
    F.count("*").alias("chamados"),
    F.countDistinct("motivo").alias("motivos"),
    F.countDistinct("canal").alias("canais"),
    F.sum(F.col("resolvido").cast("int")).alias("resolvidos"),
    F.sum((F.col("resolvido") == False).cast("int")).alias("nao_resolvidos"),
    F.sum(F.col("resolvido").isNull().cast("int")).alias("resolvido_vazio"),
    F.sum((~F.col("atendente_identificado")).cast("int")).alias("atendente_nao_identificado"),
    F.sum((~F.col("cliente_encontrado")).cast("int")).alias("cliente_nao_encontrado"),
    F.sum(F.col("tempo_espera_seg_origem").isNull().cast("int")).alias("tempo_espera_nulo"),
    F.sum(F.col("tempo_atendimento_seg_origem").isNull().cast("int")).alias("tempo_atendimento_nulo"),
)
display(resumo)

chamados,motivos,canais,resolvidos,nao_resolvidos,resolvido_vazio,atendente_nao_identificado,cliente_nao_encontrado,tempo_espera_nulo,tempo_atendimento_nulo
131838,14,4,66057,63767,2014,764,1330,75588,4463


In [0]:
# Conferência visual: devem aparecer 4 canais e 14 motivos, sem grafias soltas
display(s.groupBy("canal", "tipo_canal").count().orderBy("canal"))
display(s.groupBy("motivo").count().orderBy(F.desc("count")))

canal,tipo_canal,count
ATENDIMENTO ESPECIALIZADO,Humano,17091
ATENDIMENTO INICIAL,Humano,39697
CHATBOT,Autosservico,35517
URA,Autosservico,39533


motivo,count
Consulta de fatura,16684
Problema com aplicativo,14333
Consulta de limite,13906
Bloqueio de cartão,13128
Compra não autorizada,12991
Renegociação de dívida,12377
Contestação de fatura,11139
Desbloqueio de cartão,8810
Cancelamento de conta,5914
Consulta de contrato,5540
